In [ ]:
import torch
from torch import nn
from torch import optim
from torch.nn.utils import parameters_to_vector
from torch.nn.functional import gelu
from torchvision import datasets, transforms

import torchattacks

import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

from copy import deepcopy
from itertools import permutations

from helpers import *

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

In [ ]:
%config InlineBackend.figure_format = 'png'
device = "mps"

# Use this for Windows
device = "cuda" if torch.cuda.is_available() else "cpu"

pio.renderers.default = "notebook"  # or "vscode", "iframe", "browser" depending on your setup

# Visiualize Dataset

In [ ]:
data = iter(hypersphere_data(3, 4, 300, batch_size = 150, seed = 10))
pt, lab = next(data)
pt2, lab2 = next(data)
# pt3, lab3 = next(data)
pt.type(), lab.type()

In [ ]:
# data flattened to 2D for plotting

plt.figure(figsize=(3,3))

plt.scatter(pt[lab == 0, 0], pt[lab == 0, 1], s=10)
plt.scatter(pt[lab == 1, 0], pt[lab == 1, 1], s=10)
plt.scatter(pt[lab == 2, 0], pt[lab == 2, 1], s=10)
plt.scatter(pt[lab == 3, 0], pt[lab == 3, 1], s=10)

plt.xticks([])
plt.yticks([])
plt.show()

In [ ]:
# 3d plot of the hypersphere data

fig = px.scatter_3d(
    x=pt[:, 0],          # First column for X
    y=pt[:, 1],          # Second column for Y
    z=pt[:, 2],          # Third column for Z
    color=lab.detach().numpy().astype(str),  # Color by label, detach() in case tensors are attached to a computation graph
    labels={'color': 'Class'},
    title="Hypersphere Embedding Space",
    opacity=0.7
)

# 3. Force all markers to be small
fig.update_traces(marker_size=3.5)
fig.show()

# Functions and Classes
defualt with 2d input and 3 classes (output)

In [ ]:
# this class is used for the train_model_generator to use both yield and returns efficiently
# yield is for animations return is for everything else
class GeneratorWithReturn:
    """Wraps a generator so you can iterate it normally, and afterwards
    access whatever it `return`-ed via .value"""
    def __init__(self, gen):
        self.gen = gen
        self.value = None

    def __iter__(self):
        self.value = yield from self.gen

In [ ]:
def get_grid(input_dim, num_points_per_dim=100):

    #Classify on 100^n grid, contrained to a input_dim hypersphere
    n_arrays = [np.linspace(-1, 1, num_points_per_dim) for _ in range(input_dim)]
    grid = np.meshgrid(*n_arrays, indexing='ij')

    # keep points within the unit hypersphere
    within_radius = np.sum(np.array(grid)**2, axis=0) <= 1
    grid = [g[within_radius] for g in grid]
    # grid is a list of input_dim arrays, each of shape (num_points,) inside the hypersphere

    # stack the arrays to create a 2D array of shape (num_points, input_dim)
    X = torch.Tensor(np.column_stack(grid)).to(device)
    return X

In [ ]:
def shanon_stability(cat_all, output_dim):
    # cat_all.shape = (len(X), output_dim)
    # get % of a point being classified as a class
    cat_all = cat_all / cat_all.sum(dim=1).unsqueeze(1)

    # stability of each point on grid from Shannon formula (len(X), 1)
    # elementwise mult -> cat_all * torch.log(cat_all + 1e-10) 
    stability = 1 + torch.sum(cat_all * torch.log(cat_all + 1e-10) / np.log(output_dim), dim=1)

    return stability.detach().cpu().numpy()

In [ ]:
# train_model_gen func
# this is now a generator becuase I used "yield" to help create animations
def train_model_gen(seed, hidden_dim, n_epochs, batch_size, amount_data, 
                input_dim=2, output_dim=3, num_points_per_dim=100):
    # Load data
    train_loader = hypersphere_data(input_dim, output_dim, amount_data, 
                                    batch_size = batch_size, seed = seed)
    
    torch.manual_seed(seed)
    
    # create MLP model with the correct # dim and nodes (using GELU)
    model = MLP((input_dim, *hidden_dim, output_dim), bias=True, activation = nn.GELU)
    model.to(device)
    
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-1)
    
    # Get points on a grid in the input space to visualize decision boundaries 
    # dim: () fixed, not generated data
    X = get_grid(input_dim, num_points_per_dim=num_points_per_dim)

    cat_last = None # Keep track of decisions last iteration
    perc_change = [] # This is a list, decision % change from last to next iteration
    # Keep track of accumulating decisions over iterations
    cat_all = torch.zeros(len(X), output_dim).to(device) 
    # loss_history = [] # Use for loss over time graph

    # will iterate for n_epochs*(amount_data//batch_size)
    for _ in range(n_epochs):
        for data, target in train_loader:
            data, target = data.to(device), target.to(device)

            # outputs and loss uses data from the hypersphere_data (generated with seed)
            outputs = model(data)
            loss = criterion(outputs, target)
    
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            # loss_history.append(loss.item()) # append loss history

            # cat (to calc % change) uses fixed grid in space
            with torch.no_grad():
                # cat_next is the predicted class (values are 0,1,2,...,output_dim) for each point in the grid X, (len(X),)
                cat_next = model(X).argmax(dim=1)
                if cat_last != None:
                    # Track the percentage of predictions that changed category compared to last time
                    # list of floats, length = n_epochs * (amount_data // batch_size) - 1
                    perc_change.append( (cat_last != cat_next).sum().item() / len(X) )
                cat_last = cat_next

                # Add 1 to the col corresponding to the predicted class for each point in the grid X
                # ie sum across all columns for one row will equal the number of iteration (predictions made from training)
                for i in range(output_dim):
                    cat_all[cat_next == i, i] += 1
                
                # decision stability
                stab = shanon_stability(cat_all, output_dim)
                yield stab, cat_next

    return model, perc_change, stab#, loss_history

In [ ]:
# train_model_gen func with cat_all as queue
# everything is the same as train_model_gen other than comments
from collections import deque
def train_model_cat100(seed, hidden_dim, n_epochs, batch_size, amount_data, 
                input_dim=2, output_dim=3, num_points_per_dim=100, history_limit = 100):
    train_loader = hypersphere_data(input_dim, output_dim, amount_data, 
                                    batch_size = batch_size, seed = seed)    
    torch.manual_seed(seed)
    
    model = MLP((input_dim, *hidden_dim, output_dim), bias=True, activation = nn.GELU)
    model.to(device)
    
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-1)

    X = get_grid(input_dim, num_points_per_dim=num_points_per_dim)

    cat_last = None 
    perc_change = [] 
    predictions_history = deque(maxlen=history_limit)
    cat_all = torch.zeros(len(X), output_dim).to(device)
    # loss_history = []

    for _ in range(n_epochs):
        for data, target in train_loader:
            data, target = data.to(device), target.to(device)

            outputs = model(data)
            loss = criterion(outputs, target)
    
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            # loss_history.append(loss.item())

            with torch.no_grad():
                cat_next = model(X).argmax(dim=1)
                if cat_last != None:
                    perc_change.append( (cat_last != cat_next).sum().item() / len(X) )
                cat_last = cat_next

                # instead of using this
                # for i in range(output_dim):
                #     cat_all[cat_next == i, i] += 1
                
                # Append the current predictions to the history queue
                # (Moving to CPU prevents GPU memory leaks over time if X is huge)
                predictions_history.append(cat_next.cpu())  

                # initialize 'cat_all' to rebuild
                cat_all = torch.zeros((len(X), output_dim), dtype=torch.int32)
                
                # Stack the history along a new dimension to get shape (history_len, len(X))
                history_tensor = torch.stack(list(predictions_history))
                
                # Use scatter to count occurrences efficiently across the rolling window
                # scatter_add_ acts like a highly optimized batch version of the for-loop
                # add 1 at the index given by the history_tensor.T (add one history_len times at appropriate index)
                ones = torch.ones_like(history_tensor.transpose(0, 1), dtype=torch.int32)
                cat_all.scatter_add_(dim=1, index=history_tensor.transpose(0, 1), src=ones) # src = source

                # Calculate your stability based on the rolling 100-step history
                stab = shanon_stability(cat_all, output_dim)

                yield stab, cat_next

    return model, perc_change, stab#, loss_history

In [ ]:
# train_model_cat100 but with loss_hist return
def train_model_loss(seed, hidden_dim, n_epochs, batch_size, amount_data, 
                input_dim=2, output_dim=3, num_points_per_dim=100, history_limit = 100):
    train_loader = hypersphere_data(input_dim, output_dim, amount_data, 
                                    batch_size = batch_size, seed = seed)    
    torch.manual_seed(seed)
    
    model = MLP((input_dim, *hidden_dim, output_dim), bias=True, activation = nn.GELU)
    model.to(device)
    
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-1)

    X = get_grid(input_dim, num_points_per_dim=num_points_per_dim)

    cat_last = None 
    perc_change = [] 
    predictions_history = deque(maxlen=history_limit)
    cat_all = torch.zeros(len(X), output_dim).to(device)
    loss_history = []

    for _ in range(n_epochs):
        for data, target in train_loader:
            data, target = data.to(device), target.to(device)

            outputs = model(data)
            loss = criterion(outputs, target)
    
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            loss_history.append(loss.item())

            with torch.no_grad():
                cat_next = model(X).argmax(dim=1)
                if cat_last != None:
                    perc_change.append( (cat_last != cat_next).sum().item() / len(X) )
                cat_last = cat_next

                # instead of using this
                # for i in range(output_dim):
                #     cat_all[cat_next == i, i] += 1
                
                # Append the current predictions to the history queue
                # (Moving to CPU prevents GPU memory leaks over time if X is huge)
                predictions_history.append(cat_next.cpu())  

                # initialize 'cat_all' to rebuild
                cat_all = torch.zeros((len(X), output_dim), dtype=torch.int32)
                
                # Stack the history along a new dimension to get shape (history_len, len(X))
                history_tensor = torch.stack(list(predictions_history))
                
                # Use scatter to count occurrences efficiently across the rolling window
                # scatter_add_ acts like a highly optimized batch version of the for-loop
                # add 1 at the index given by the history_tensor.T (add one history_len times at appropriate index)
                ones = torch.ones_like(history_tensor.transpose(0, 1), dtype=torch.int32)
                cat_all.scatter_add_(dim=1, index=history_tensor.transpose(0, 1), src=ones) # src = source

                # Calculate your stability based on the rolling 100-step history
                stab = shanon_stability(cat_all, output_dim)

                yield stab, cat_next

    return model, perc_change, stab, loss_history

In [ ]:
# animate_training func
import matplotlib.animation as animation
from matplotlib.colors import ListedColormap

def animate_training(gen,# seed, hidden_dim, n_epochs, batch_size, amount_data,
                      input_dim=2, output_dim=3, num_points_per_dim=100,
                      interval=50, overlay_seed=10):
    # interval = delay time in milisec for each frame
    # overlay_seed = seed for overlay scatter of another generated hypersphere data ()

    X = get_grid(input_dim, num_points_per_dim=num_points_per_dim)
    X_np = X.cpu().numpy() # turn grid into numpy array

    # Put generator from train_model_ in wrapper
    # gen = train_model_gen(seed, hidden_dim, n_epochs, batch_size, amount_data,
    #                    input_dim, output_dim, num_points_per_dim)
    wrapped = GeneratorWithReturn(gen)

    fig, (ax_bound, ax_stab) = plt.subplots(1, 2, figsize=(14, 6))

    # Get colors for the output classes
    class_cmap = ListedColormap(plt.cm.tab10.colors[:output_dim])

    # Initialize plots
    # Decision boundary scatter (colors updated each frame) 
    # use colormap instead of plotting 3 times...
    scat_bound = ax_bound.scatter(X_np[:, 0], X_np[:, 1], s=1, marker="s",
                                   c=np.zeros(len(X_np)), cmap=class_cmap,
                                   vmin=0, vmax=output_dim - 1)
    ax_bound.set_title("Decision boundary")
    ax_bound.set_xticks([]); ax_bound.set_yticks([])

    # Stability scatter 
    scat_stab = ax_stab.scatter(X_np[:, 0], X_np[:, 1], s=1, marker="s",
                                 c=np.zeros(len(X_np)), cmap="Spectral",
                                 vmin=0, vmax=1)
    ax_stab.set_title("Decision stability")
    ax_stab.figure.colorbar(scat_stab, ax=ax_stab)
    ax_stab.set_xticks([]); ax_stab.set_yticks([])

    # Create overlay scatter plot from data points generated by a different seed hypersphere_data
    # Plot shows the true labels for reference, nothing to do with MLP training stuff
    pt, lab = next(iter(hypersphere_data(input_dim, output_dim, 150,
                                          batch_size=150,
                                          seed=overlay_seed)))
    pt_np = pt.cpu().numpy() # turn into np
    lab_np = lab.cpu().numpy()

    # overlay the data in ax_bound, ax_stab for each output class
    for ax in (ax_bound, ax_stab):
        for i in range(output_dim):
            mask = lab_np == i
            ax.scatter(pt_np[mask, 0], pt_np[mask, 1], s=20, marker="s",
                       alpha=0.8, edgecolor="white", linewidth=0.7,
                       color=class_cmap(i))

    # function for updating the stab and cat_next each frame
    # which you can iterate using the generator from wrapped 
    # (produced by "yield" in the train_model_gen)
    def update(frame):
        stab, cat_next = frame
        cat_np = cat_next.cpu().numpy()
        # if stab is a tensor, convert properly; otherwise just wrap it as an array
        stab_np = stab.cpu().numpy() if hasattr(stab, "cpu") else np.asarray(stab) # pretty sure it's a tensor bit just incase

        scat_bound.set_array(cat_np)
        scat_stab.set_array(stab_np)
        return scat_bound, scat_stab

    # Create animation object
    ani = animation.FuncAnimation(
        fig, update, frames=wrapped, interval=interval,
        blit=False, cache_frame_data=False
    )
    plt.close(fig)

    return ani, wrapped

In [ ]:
def plot_decision_boundary_2D(model, title="", output_dim=3, num_points_per_dim=100):
    X = get_grid(input_dim=2, num_points_per_dim=num_points_per_dim)

    # SHAPES:
    # model.net[1].weight.T.shape = (3, hidden_dim)
    # model.net[1].bias.unsqueeze(0).shape = (1, hidden_dim)
        # out1: (N, 3) @ (3, hidden_dim) + (1, hidden_dim) -> (N, hidden_dim) ie N data points, each with hidden_dim features
    # model.net[2] is the GeLU\ReLU activation function, shape preserved
    # model.net[3].weight.T.shape = (hidden_dim, output_dim) 
    # model.net[3].bias.unsqueeze(0).shape = (1, output_dim)
        # out2: (N, hidden_dim) @ (hidden_dim, output_dim) + (1, output_dim) -> (N, output_dim) 
    # out1 = model.net[2](X @ model.net[1].weight.T + model.net[1].bias.unsqueeze(0))
    # out2 = out1 @ model.net[3].weight.T + model.net[3].bias.unsqueeze(0)

    # each of N data points has scores for each class (output_dim = # of classes), largest number is the predicted class
    # cat.shape = (N,) where each entry is the predicted class (index of largest output_dim) for that data point
    # cat = out2.argmax(dim=1)
    cat = model(X).argmax(dim=1)
    
    plt.figure(figsize = (2,2))
    # plot each class in a different color
    for i in range(output_dim):
        plt.scatter(X[cat == i,0], X[cat == i,1], s=1, marker="s", alpha=0.8)

    plt.title("Decision Boundaries\n" + title)
    plt.show()

In [ ]:
def plot_stability_2D(stab, num_points_per_dim=100, title="", output_dim=3, seed=10):
    X = get_grid(input_dim=2, num_points_per_dim=num_points_per_dim)
    # cat = model(X).argmax(dim=1)

    plt.figure(figsize = (4,3))
    # plot a heat map based on the stability in the input space
    plt.scatter(X[:,0], X[:,1], s=1, c=stab, cmap="Spectral", alpha=0.9)
    plt.colorbar()


    pt, lab = next(iter(hypersphere_data(2, output_dim, 150, batch_size = 150, seed = seed)))
    for i in range(output_dim):
        plt.scatter(pt[lab == i, 0], pt[lab == i, 1], s=10, marker="s", alpha=0.8, edgecolor="white")
    
    plt.title("Stability from Shannon\n" + title)
    plt.show()

    print(f"Mean Stability: {np.mean(stab)}")

In [ ]:
def plot_stability_3D(stab, num_points_per_dim=50, title="", output_dim=3, seed=10):
    X = get_grid(input_dim=3, num_points_per_dim=num_points_per_dim)
    X_np = X.detach().cpu().numpy()
    stab_np = stab.detach().cpu().numpy() if hasattr(stab, "detach") else np.asarray(stab)

    fig = px.scatter_3d(x=X_np[:, 0], y=X_np[:, 1], z=X_np[:, 2],
        color=stab_np,
        color_continuous_scale="Spectral",
        opacity=0.1,
        labels={'x': 'X1', 'y': 'X2', 'z': 'X3'},
        title="Stability from Shannon<br>" + title
    )


    pt, lab = next(iter(hypersphere_data(3, output_dim, 150, batch_size=150, seed=seed)))
    pt_np = pt.detach().cpu().numpy()
    lab_np = lab.detach().cpu().numpy()

    for i in range(output_dim):
        cluster_mask = (lab_np == i)
        fig.add_trace(
            go.Scatter3d(
                x=pt_np[cluster_mask, 0], y=pt_np[cluster_mask, 1], z=pt_np[cluster_mask, 2],
                mode='markers',
                name=f'Cluster {i}',
                marker=dict(size=6, symbol='square', opacity=0.5, line=dict(color='white', width=1))
            )
        )

    fig.update_traces(marker=dict(size=2))
    fig.update_layout(width=700, height=450, coloraxis_colorbar=dict(title="Stability", x=1.15,),
                      legend=dict(yanchor="top", y=0.99, xanchor="left", x=1.25 ))
    
    fig.show()

    print(f"Mean Stability: {np.mean(stab)}")

In [ ]:
def plot_perc_change(perc_change, title="model0"):
    plt.figure(figsize = (8,2))
    plt.plot([100*x for x in perc_change])
    plt.xlabel("Training iteration")
    plt.ylabel("Decisions changed (%)")
    plt.title(title)
    plt.show()

# Simple Visuals From Training

In [ ]:
# # 2D grid for decision boundary visualization
# # checking outputs (just printing shapes and stuff)

# x = np.linspace(-1,1,100)
# y = np.linspace(-1,1,100)
# x, y = np.meshgrid(x, y)

# within_radius = x ** 2 + y ** 2 <= 1
# x = x[within_radius]
# y = y[within_radius]
# print(x)
# print(x.shape)

# X = torch.Tensor(np.append(x,y).reshape(2,-1)).T.to(device)
# print(X[:,0])

# (X @ model2D.net[1].weight.T).shape
# model2D.net[3].weight.T.shape
# model2D.net[1].bias.unsqueeze(0).shape
# model2D.net[2](X @ model2D.net[1].weight.T + model2D.net[1].bias) @ model2D.net[3].weight.T + model2D.net[3].bias.unsqueeze(0)
# a = (model2D.net[2](X @ model2D.net[1].weight.T + model2D.net[1].bias) @ model2D.net[3].weight.T + model2D.net[3].bias.unsqueeze(0)).argmax(dim=1)

# plt.scatter(x[a == 0], y[a == 0], s=1, marker="s", alpha=0.8)
# plt.scatter(x[a == 1], y[a == 1], s=1, marker="s", alpha=0.8)
# plt.scatter(x[a == 2], y[a == 2], s=1, marker="s", alpha=0.8)


In [ ]:
gen2D = train_model_gen(seed = 1, hidden_dim = (8,), n_epochs = 100, amount_data = 100, batch_size = 5, num_points_per_dim=150)
wrapper2D = GeneratorWithReturn(gen2D)

# discard intermediate (stab, cat_next) frames
for _ in wrapper2D: pass
# get final return values
model2D, perc_change2D, stab2D = wrapper2D.value


In [ ]:
plot_perc_change(perc_change2D, title="2D, 3Classes")
plot_decision_boundary_2D(model2D, title="2:8:3")
plot_stability_2D(stab2D, title="2:8:3", num_points_per_dim=150)

# Model Structure 2_16_16_3
testing seed, n_epochs, amount_data, batch_size

In [ ]:
# (function) def train_model_gen(
#  seed: Any,
#  hidden_dim: Any,
#  n_epochs: Any,
#  batch_size: Any,
#  amount_data: Any,
#  input_dim: int = 2,
#  output_dim: int = 3,
#  num_points_per_dim: int = 100
# ) -> Generator[tuple[ndarray, Any], Any, tuple[MLP, list, Unbound | ndarray]]

# (function) def animate_training(
#  gen: Any,
#  input_dim: int = 2,
#  output_dim: int = 3,
#  num_points_per_dim: int = 100,
#  interval: int = 50,
#  overlay_seed: int = 10
# ) -> tuple[FuncAnimation, GeneratorWithReturn]

In [ ]:
gen = train_model_gen(seed=0, hidden_dim=[16,16], n_epochs=30, batch_size=30, amount_data=300, num_points_per_dim=150)
ani, wrapped = animate_training(gen, num_points_per_dim=150)

In [ ]:
# from IPython.display import HTML
# HTML(ani.to_jshtml())
ani.save('ani/animation_2_16_16_3_seed0_cat_all.mp4', writer='ffmpeg', fps=15)
from IPython.display import Video
Video('ani/animation_2_16_16_3_seed0_cat_all.mp4', embed=True)

In [ ]:
model, perc_change, stab = wrapped.value
plot_perc_change(perc_change, "2:16:16:3")
plot_decision_boundary_2D(model, title="2:16:16:3")
plot_stability_2D(stab, num_points_per_dim=150, title="2:16:16:3")

Try same thing with stability calculated with last 100 training decisions (cat100)

In [ ]:
gen = train_model_cat100(seed=0, hidden_dim=[16,16], n_epochs=30, batch_size=30, amount_data=300, num_points_per_dim=150)
ani, wrapped = animate_training(gen, num_points_per_dim=150)
# from IPython.display import HTML
# HTML(ani.to_jshtml())
ani.save('ani/animation_2_16_16_3_seed0_cat100.mp4', writer='ffmpeg', fps=15)
from IPython.display import Video
Video('ani/animation_2_16_16_3_seed0_cat100.mp4', embed=True)

Try cat100 with seed5


In [ ]:
gen = train_model_cat100(seed = 5, hidden_dim = [16,16], n_epochs = 30, amount_data = 300, batch_size = 30, num_points_per_dim=150)
wrapper = GeneratorWithReturn(gen)

# discard intermediate (stab, cat_next) frames
for _ in wrapper: pass
# get final return values
model, perc_change, stab = wrapper.value

# plot_perc_change(perc_change, "2:16:16:3")
# plot_decision_boundary_2D(model, title="2:16:16:3")
plot_stability_2D(stab, num_points_per_dim=150, title="2:16:16:3")

try cat100 with seed71

In [ ]:
gen = train_model_cat100(seed = 71, hidden_dim = [16,16], n_epochs = 30, amount_data = 300, batch_size = 30, num_points_per_dim=150)
wrapper = GeneratorWithReturn(gen)

# discard intermediate (stab, cat_next) frames
for _ in wrapper: pass
# get final return values
model, perc_change, stab = wrapper.value

plot_stability_2D(stab, num_points_per_dim=150, title="2:16:16:3")

cat100 w seed42

In [ ]:
gen = train_model_cat100(seed = 42, hidden_dim = [16,16], n_epochs = 30, amount_data = 300, batch_size = 30, num_points_per_dim=150)
wrapper = GeneratorWithReturn(gen)

# discard intermediate (stab, cat_next) frames
for _ in wrapper: pass
# get final return values
model, perc_change, stab = wrapper.value

plot_stability_2D(stab, num_points_per_dim=150, title="2:16:16:3")

Model seems to be performing pretty consistenly with different seeds. Try decreasing the amount of data change batch size and epochs (training iterations).

cat100, seed5, n_epochs = 10, amount_data = 100, batch_size = 50

In [ ]:
gen = train_model_cat100(seed = 5, hidden_dim = [16,16], n_epochs = 10, amount_data = 100, batch_size = 50, num_points_per_dim=150)
wrapper = GeneratorWithReturn(gen)

# discard intermediate (stab, cat_next) frames
for _ in wrapper: pass
# get final return values
model, perc_change, stab = wrapper.value

plot_stability_2D(stab, num_points_per_dim=150, title="2:16:16:3")

gen seed5, n_epochs = 100, amount_data = 100, batch_size = 100

In [ ]:
gen = train_model_gen(seed = 5, hidden_dim = [16,16], n_epochs = 100, amount_data = 100, batch_size = 100, num_points_per_dim=150)
wrapper = GeneratorWithReturn(gen)

# discard intermediate (stab, cat_next) frames
for _ in wrapper: pass
# get final return values
model, perc_change, stab = wrapper.value

plot_stability_2D(stab, num_points_per_dim=150, title="2:16:16:3")

cat100, seed tried many, n_epochs = 30, amount_data = 100, batch_size = 20 

pretty good combo for faster training (watch color bar scale)

In [ ]:
gen = train_model_cat100(seed = 12, hidden_dim = [16,16], n_epochs = 30, amount_data = 100, batch_size = 20, num_points_per_dim=150)
wrapper = GeneratorWithReturn(gen)

# discard intermediate (stab, cat_next) frames
for _ in wrapper: pass
# get final return values
model, perc_change, stab = wrapper.value

plot_stability_2D(stab, num_points_per_dim=150, title="2:16:16:3")

# Model Structure: 2_4_3
(slighly worse)

In [ ]:
gen = train_model_gen(seed=0, hidden_dim=(4,), n_epochs=30, batch_size=30, amount_data=300, num_points_per_dim=150)
ani, wrapped = animate_training(gen, num_points_per_dim=150)# from IPython.display import HTML
# HTML(ani.to_jshtml())
ani.save('ani/animation_2_4_3_seed0_cat_all.mp4', writer='ffmpeg', fps=15)
from IPython.display import Video
Video('ani/animation_2_4_3_seed0_cat_all.mp4', embed=True)

In [ ]:
gen = train_model_cat100(seed=0, hidden_dim=(4,), n_epochs=30, batch_size=30, amount_data=300, num_points_per_dim=150)
ani, wrapped = animate_training(gen, num_points_per_dim=150)
# from IPython.display import HTML
# HTML(ani.to_jshtml())
ani.save('ani/animation_2_4_3_seed0_cat100.mp4', writer='ffmpeg', fps=15)
from IPython.display import Video
Video('ani/animation_2_4_3_seed0_cat100.mp4', embed=True)

# Other Structures
with less data

In [ ]:
gen = train_model_cat100(seed = 50, hidden_dim = (4,), n_epochs = 30, amount_data = 100, batch_size = 20, num_points_per_dim=150)
wrapper = GeneratorWithReturn(gen)

# discard intermediate (stab, cat_next) frames
for _ in wrapper: pass
# get final return values
model, perc_change, stab = wrapper.value

plot_stability_2D(stab, num_points_per_dim=150, title="2:4:3")

In [ ]:
gen = train_model_cat100(seed = 50, hidden_dim = (4,4), n_epochs = 30, amount_data = 100, batch_size = 20, num_points_per_dim=150)
wrapper = GeneratorWithReturn(gen)

# discard intermediate (stab, cat_next) frames
for _ in wrapper: pass
# get final return values
model, perc_change, stab = wrapper.value

plot_stability_2D(stab, num_points_per_dim=150, title="2:4:4:3")

In [ ]:
gen = train_model_cat100(seed = 50, hidden_dim = (16,16), n_epochs = 30, amount_data = 100, batch_size = 20, num_points_per_dim=150)
wrapper = GeneratorWithReturn(gen)

# discard intermediate (stab, cat_next) frames
for _ in wrapper: pass
# get final return values
model, perc_change, stab = wrapper.value

plot_stability_2D(stab, num_points_per_dim=150, title="2:16:16:3")

In [ ]:
gen = train_model_cat100(seed = 50, hidden_dim = (16,), n_epochs = 30, amount_data = 100, batch_size = 20, num_points_per_dim=150)
wrapper = GeneratorWithReturn(gen)

# discard intermediate (stab, cat_next) frames
for _ in wrapper: pass
# get final return values
model, perc_change, stab = wrapper.value

plot_stability_2D(stab, num_points_per_dim=150, title="2:16:16:3")

In [ ]:
gen = train_model_cat100(seed = 50, hidden_dim = (3,), n_epochs = 30, amount_data = 100, batch_size = 20, num_points_per_dim=150)
wrapper = GeneratorWithReturn(gen)

# discard intermediate (stab, cat_next) frames
for _ in wrapper: pass
# get final return values
model, perc_change, stab = wrapper.value

plot_stability_2D(stab, num_points_per_dim=150, title="2:16:16:3")

In [ ]:
gen = train_model_cat100(seed = 50, hidden_dim = (3,3), n_epochs = 30, amount_data = 100, batch_size = 20, num_points_per_dim=150)
wrapper = GeneratorWithReturn(gen)

# discard intermediate (stab, cat_next) frames
for _ in wrapper: pass
# get final return values
model, perc_change, stab = wrapper.value

plot_stability_2D(stab, num_points_per_dim=150, title="2:16:16:3")

In [ ]:
gen = train_model_cat100(seed = 50, hidden_dim = (100,), n_epochs = 30, amount_data = 100, batch_size = 20, num_points_per_dim=150)
wrapper = GeneratorWithReturn(gen)

# discard intermediate (stab, cat_next) frames
for _ in wrapper: pass
# get final return values
model, perc_change, stab = wrapper.value

plot_stability_2D(stab, num_points_per_dim=150, title="2:16:16:3")

In [ ]:
gen = train_model_cat100(seed = 50, hidden_dim = (50,), n_epochs = 30, amount_data = 100, batch_size = 20, num_points_per_dim=150)
wrapper = GeneratorWithReturn(gen)

# discard intermediate (stab, cat_next) frames
for _ in wrapper: pass
# get final return values
model, perc_change, stab = wrapper.value

plot_stability_2D(stab, num_points_per_dim=150, title="2:16:16:3")

In [ ]:
gen = train_model_cat100(seed = 50, hidden_dim = (50,50), n_epochs = 30, amount_data = 100, batch_size = 20, num_points_per_dim=150)
wrapper = GeneratorWithReturn(gen)

# discard intermediate (stab, cat_next) frames
for _ in wrapper: pass
# get final return values
model, perc_change, stab = wrapper.value

plot_stability_2D(stab, num_points_per_dim=150, title="2:16:16:3")

In [ ]:
gen = train_model_cat100(seed = 50, hidden_dim = (50,2), n_epochs = 30, amount_data = 100, batch_size = 20, num_points_per_dim=150)
wrapper = GeneratorWithReturn(gen)

# discard intermediate (stab, cat_next) frames
for _ in wrapper: pass
# get final return values
model, perc_change, stab = wrapper.value

plot_stability_2D(stab, num_points_per_dim=150, title="2:16:16:3")

In [ ]:
gen = train_model_cat100(seed = 50, hidden_dim = (100,100), n_epochs = 30, amount_data = 100, batch_size = 20, num_points_per_dim=150)
wrapper = GeneratorWithReturn(gen)

# discard intermediate (stab, cat_next) frames
for _ in wrapper: pass
# get final return values
model, perc_change, stab = wrapper.value

plot_stability_2D(stab, num_points_per_dim=150, title="2:16:16:3")

Single layers seems to work better, at least for when the input is 2D and output is 3 classes.

For single layers, it seems like the more nodes the better.

Double layers don't work very well, and increasing the number of nodes doesn't always make it better. (100,50) was better than (100,100), but (50,50) is better than (50,2).

# output_dim>=4, different structures

more nodes and more hidden dim doesn't always mean better... but for out_dim=4 i think generally more nodes for single layers get better results

In [ ]:
gen = train_model_cat100(seed = 50, hidden_dim = (6,18,48), output_dim=4, n_epochs = 30, amount_data = 100, batch_size = 20, num_points_per_dim=150)
wrapper = GeneratorWithReturn(gen)

for _ in wrapper: pass
model, perc_change, stab = wrapper.value

plot_stability_2D(stab, output_dim=4, num_points_per_dim=150, title="2:6:18:48:4")

In [ ]:
gen = train_model_gen(seed = 50, hidden_dim = (50,2), output_dim=4, n_epochs = 30, amount_data = 100, batch_size = 20, num_points_per_dim=150)
wrapper = GeneratorWithReturn(gen)

# discard intermediate (stab, cat_next) frames
for _ in wrapper: pass
# get final return values
model, perc_change, stab = wrapper.value

plot_stability_2D(stab, output_dim=4, num_points_per_dim=150, title="2:50:2:4")

In [ ]:
gen = train_model_cat100(seed = 108, hidden_dim = (50,), output_dim=4, n_epochs = 30, amount_data = 100, batch_size = 20, num_points_per_dim=150)
wrapper = GeneratorWithReturn(gen)

# discard intermediate (stab, cat_next) frames
for _ in wrapper: pass
# get final return values
model, perc_change, stab = wrapper.value

plot_stability_2D(stab, output_dim=4, num_points_per_dim=150, title="2:50:4")

In [ ]:
gen = train_model_cat100(seed = 50, hidden_dim = (4,4), output_dim=4, n_epochs = 30, amount_data = 100, batch_size = 20, num_points_per_dim=150)
wrapper = GeneratorWithReturn(gen)

# discard intermediate (stab, cat_next) frames
for _ in wrapper: pass
# get final return values
model, perc_change, stab = wrapper.value

plot_stability_2D(stab, output_dim=4, num_points_per_dim=150, title="2:50:4")

In [ ]:
gen = train_model_cat100(seed = 50, hidden_dim = (4,), output_dim=4, n_epochs = 30, amount_data = 100, batch_size = 20, num_points_per_dim=150)
wrapper = GeneratorWithReturn(gen)

# discard intermediate (stab, cat_next) frames
for _ in wrapper: pass
# get final return values
model, perc_change, stab = wrapper.value

plot_stability_2D(stab, output_dim=4, num_points_per_dim=150, title="2:4:4")

In [ ]:
gen = train_model_cat100(seed = 108, hidden_dim = (16,16), output_dim=4, n_epochs = 30, amount_data = 100, batch_size = 20, num_points_per_dim=150)
wrapper = GeneratorWithReturn(gen)

# discard intermediate (stab, cat_next) frames
for _ in wrapper: pass
# get final return values
model, perc_change, stab = wrapper.value

plot_stability_2D(stab, output_dim=4, num_points_per_dim=150, title="2:16:16:4")

In [ ]:
gen = train_model_cat100(seed = 108, hidden_dim = (16,), output_dim=4, n_epochs = 30, amount_data = 100, batch_size = 20, num_points_per_dim=150)
wrapper = GeneratorWithReturn(gen)

# discard intermediate (stab, cat_next) frames
for _ in wrapper: pass
# get final return values
model, perc_change, stab = wrapper.value

plot_stability_2D(stab, output_dim=4, num_points_per_dim=150, title="2:16:4")

In [ ]:
# animation
gen = train_model_cat100(seed=0, hidden_dim=(16,), output_dim=4, n_epochs=30, batch_size=30, amount_data=300, num_points_per_dim=150)
ani, wrapped = animate_training(gen, output_dim=4, num_points_per_dim=150)
# from IPython.display import HTML
# HTML(ani.to_jshtml())
ani.save('ani/animation_2_4_3_seed0_cat100.mp4', writer='ffmpeg', fps=15)
from IPython.display import Video
Video('ani/animation_2_4_3_seed0_cat100.mp4', embed=True)

Single layer still works better for out_dim=4

output_dim=5 : change the epochs and batch size

In [ ]:
gen = train_model_loss(seed = 50, hidden_dim = (16,), output_dim=5, n_epochs = 80, amount_data = 100, batch_size = 20, num_points_per_dim=150)
wrapper = GeneratorWithReturn(gen)

# discard intermediate (stab, cat_next) frames
for _ in wrapper: pass
# get final return values
model, perc_change, stab, loss = wrapper.value

plot_stability_2D(stab, output_dim=5, num_points_per_dim=150, title="2:16:5")
print(f"Loss at the end of training: {loss[-1]}")

In [ ]:
gen = train_model_cat100(seed = 315, hidden_dim = (16,4), output_dim=5, n_epochs = 80, amount_data = 100, batch_size = 20, num_points_per_dim=150)
wrapper = GeneratorWithReturn(gen)

# discard intermediate (stab, cat_next) frames
for _ in wrapper: pass
# get final return values
model, perc_change, stab = wrapper.value

plot_stability_2D(stab, output_dim=5, num_points_per_dim=150, title="2:16:4:5")
print(f"Loss at the end of training: {loss[-1]}")

In [ ]:
gen = train_model_loss(seed = 315, hidden_dim = (100,), output_dim=5, n_epochs = 80, amount_data = 100, batch_size = 20, num_points_per_dim=150)
wrapper = GeneratorWithReturn(gen)

for _ in wrapper: pass
model, perc_change, stab, loss = wrapper.value

plot_stability_2D(stab, output_dim=5, num_points_per_dim=150, title="2:100:5")
print(f"Loss at the end of training: {loss[-1]}")

In [ ]:
# animation
gen = train_model_cat100(seed=74, hidden_dim=(16,4), output_dim=5, n_epochs=80, batch_size=20, amount_data=100, num_points_per_dim=150)
ani, wrapped = animate_training(gen, output_dim=5, num_points_per_dim=150)
# from IPython.display import HTML
# HTML(ani.to_jshtml())
ani.save('ani/animation_2_16_5_seed74_cat100.mp4', writer='ffmpeg', fps=15)
from IPython.display import Video
Video('ani/animation_2_16_5_seed74_cat100.mp4', embed=True)

(16,) worked the best-ish

increase data for out_dim=7

In [ ]:
gen = train_model_loss(seed = 73, hidden_dim = (16,), output_dim=7, n_epochs = 60, amount_data = 400, batch_size = 40, num_points_per_dim=150)
wrapper = GeneratorWithReturn(gen)

# discard intermediate (stab, cat_next) frames
for _ in wrapper: pass
# get final return values
model, perc_change, stab, loss = wrapper.value

plot_stability_2D(stab, output_dim=7, num_points_per_dim=150, title="2:16:7")
print(f"Loss at the end of training: {loss[-1]}")

In [ ]:
gen = train_model_loss(seed = 73, hidden_dim = (16,4), output_dim=7, n_epochs = 60, amount_data = 400, batch_size = 40, num_points_per_dim=150)
wrapper = GeneratorWithReturn(gen)

# discard intermediate (stab, cat_next) frames
for _ in wrapper: pass
# get final return values
model, perc_change, stab, loss = wrapper.value

plot_stability_2D(stab, output_dim=7, num_points_per_dim=150, title="2:16:4:7")
print(f"Loss at the end of training: {loss[-1]}")

In [ ]:
gen = train_model_loss(seed = 90, hidden_dim = (50,), output_dim=7, n_epochs = 60, amount_data = 400, batch_size = 40, num_points_per_dim=150)
wrapper = GeneratorWithReturn(gen)

# discard intermediate (stab, cat_next) frames
for _ in wrapper: pass
# get final return values
model, perc_change, stab, loss = wrapper.value

plot_stability_2D(stab, output_dim=7, num_points_per_dim=150, title="2:50:7")
print(f"Loss at the end of training: {loss[-1]}")

In [ ]:
gen = train_model_loss(seed = 90, hidden_dim = (150,), output_dim=7, n_epochs = 60, amount_data = 400, batch_size = 40, num_points_per_dim=150)
wrapper = GeneratorWithReturn(gen)

# discard intermediate (stab, cat_next) frames
for _ in wrapper: pass
# get final return values
model, perc_change, stab, loss = wrapper.value

plot_stability_2D(stab, output_dim=7, num_points_per_dim=150, title="2:150:7")
print(f"Loss at the end of training: {loss[-1]}")

In [ ]:
gen = train_model_loss(seed = 90, hidden_dim = (50,50), output_dim=7, n_epochs = 60, amount_data = 400, batch_size = 40, num_points_per_dim=150)
wrapper = GeneratorWithReturn(gen)

# discard intermediate (stab, cat_next) frames
for _ in wrapper: pass
# get final return values
model, perc_change, stab, loss = wrapper.value

plot_stability_2D(stab, output_dim=7, num_points_per_dim=150, title="2:50:50:7")
print(f"Loss at the end of training: {loss[-1]}")

(16,) or (16,4) was the best, increasing nodes doesn't result better loss or stabiltiy

out_dim=10, increase data

In [ ]:
gen = train_model_loss(seed = 90, hidden_dim = (16,4), output_dim=10, n_epochs = 60, amount_data = 400, batch_size = 40, num_points_per_dim=150)
wrapper = GeneratorWithReturn(gen)

# discard intermediate (stab, cat_next) frames
for _ in wrapper: pass
# get final return values
model, perc_change, stab, loss = wrapper.value

plot_stability_2D(stab, output_dim=7, num_points_per_dim=150, title="2:16:4:7")
print(f"Loss at the end of training: {loss[-1]}")

In [ ]:
gen = train_model_loss(seed = 90, hidden_dim = (50,50), output_dim=10, n_epochs = 60, amount_data = 400, batch_size = 40, num_points_per_dim=150)
wrapper = GeneratorWithReturn(gen)

# discard intermediate (stab, cat_next) frames
for _ in wrapper: pass
# get final return values
model, perc_change, stab, loss = wrapper.value

plot_stability_2D(stab, output_dim=7, num_points_per_dim=150, title="2:50:50:7")
print(f"Loss at the end of training: {loss[-1]}")

In [ ]:
gen = train_model_loss(seed = 90, hidden_dim = (150,), output_dim=10, n_epochs = 60, amount_data = 400, batch_size = 40, num_points_per_dim=150)
wrapper = GeneratorWithReturn(gen)

for _ in wrapper: pass
model, perc_change, stab, loss = wrapper.value

plot_stability_2D(stab, output_dim=7, num_points_per_dim=150, title="2:150:7")
print(f"Loss at the end of training: {loss[-1]}")

seems like double layers work better for higher dim (not necessarily better than single layers -- but they sometimes have prettier decision boundaries, not sure if that means the model overfits more easily for single layers???)???? but not always the more nodes the better

# In 3d...

In [ ]:
gen = train_model_cat100(seed = 50, hidden_dim = (4,), input_dim=3, output_dim=3, n_epochs = 60, amount_data = 400, batch_size = 40, num_points_per_dim=50)
wrapper = GeneratorWithReturn(gen)

for _ in wrapper: pass
model, perc_change, stab = wrapper.value

plot_stability_3D(stab, output_dim=3, num_points_per_dim=50, title="3:4:3")

In [ ]:
gen = train_model_cat100(seed = 50, hidden_dim = (16,), input_dim=3, output_dim=3, n_epochs = 60, amount_data = 400, batch_size = 40, num_points_per_dim=50)
wrapper = GeneratorWithReturn(gen)

# discard intermediate (stab, cat_next) frames
for _ in wrapper: pass
# get final return values
model, perc_change, stab = wrapper.value

plot_stability_3D(stab, output_dim=3, num_points_per_dim=50, title="3:16:3")

In [ ]:
gen = train_model_cat100(seed = 50, hidden_dim = (16,), input_dim=3, output_dim=5, n_epochs = 60, amount_data = 400, batch_size = 40, num_points_per_dim=50)
wrapper = GeneratorWithReturn(gen)

# discard intermediate (stab, cat_next) frames
for _ in wrapper: pass
# get final return values
model, perc_change, stab = wrapper.value

plot_stability_3D(stab, output_dim=5, num_points_per_dim=50, title="3:16:5")

In [ ]:
gen = train_model_loss(seed = 50, hidden_dim = (16,), input_dim=3, output_dim=10, n_epochs = 60, amount_data = 400, batch_size = 40, num_points_per_dim=50)
wrapper = GeneratorWithReturn(gen)

# discard intermediate (stab, cat_next) frames
for _ in wrapper: pass
# get final return values
model, perc_change, stab, loss = wrapper.value

plot_stability_3D(stab, output_dim=10, num_points_per_dim=50, title="3:16:10")
print(f"Loss at the end of training: {loss[-1]}")

In [ ]:
gen = train_model_loss(seed = 50, hidden_dim = (16,4), input_dim=3, output_dim=10, n_epochs = 60, amount_data = 400, batch_size = 40, num_points_per_dim=50)
wrapper = GeneratorWithReturn(gen)

# discard intermediate (stab, cat_next) frames
for _ in wrapper: pass
# get final return values
model, perc_change, stab, loss = wrapper.value

plot_stability_3D(stab, output_dim=10, num_points_per_dim=50, title="3:16:4:10")
print(f"Loss at the end of training: {loss[-1]}")

In [ ]:
gen = train_model_loss(seed = 50, hidden_dim = (50,), input_dim=3, output_dim=10, n_epochs = 100, amount_data = 1200, batch_size = 40, num_points_per_dim=50)
wrapper = GeneratorWithReturn(gen)

# discard intermediate (stab, cat_next) frames
for _ in wrapper: pass
# get final return values
model, perc_change, stab, loss = wrapper.value

plot_stability_3D(stab, output_dim=10, num_points_per_dim=50, title="3:50:10")
print(f"Loss at the end of training: {loss[-1]}")

In [ ]:
gen = train_model_loss(seed = 50, hidden_dim = (50,50), input_dim=3, output_dim=10, n_epochs = 100, amount_data = 1200, batch_size = 40, num_points_per_dim=50)
wrapper = GeneratorWithReturn(gen)

# discard intermediate (stab, cat_next) frames
for _ in wrapper: pass
# get final return values
model, perc_change, stab, loss = wrapper.value

plot_stability_3D(stab, output_dim=10, num_points_per_dim=50, title="3:50:50:10")
print(f"Loss at the end of training: {loss[-1]}")

i think (50,) or (100,) was slightly better than (16,) much better than (50,2) better than (100,2)

    But single layer still in general performs better and is cheaper? (based on both stability and loss)

# Input Space Over time

In [ ]:
gen = train_model_loss(seed=3, hidden_dim=(16,), input_dim=2, output_dim=5, n_epochs=40, amount_data=200, batch_size=20, num_points_per_dim=100)
wrapper = GeneratorWithReturn(gen)

for _ in wrapper: pass
model, perc_change, stab, loss = wrapper.value

plot_stability_2D(stab, output_dim=5, num_points_per_dim=100, title="3:16:5")
print(f"Loss at end of training: {loss[-1]}")

In [ ]:
print(model.net)

In [ ]:
# data, (#data, in_dim)
X = get_grid(input_dim=2)

# first layer, (#data,in_dim) @ (in_dim, 20) + (1, 20)       add bias to each #data
out1 = X @ model.net[1].weight.T + model.net[1].bias.unsqueeze(0) 
# GeLU 
out2 = model.net[2](out1)
# sec layer, (#data,20) @ (20, 20) + (1, 20)
out3 = out2 @ model.net[3].weight.T + model.net[3].bias.unsqueeze(0)
# # GeLU 
# out4 = model.net[4](out3)
# # output, (#data,20) @ (20, 5) + (1, 5)
# out5 = out4 @ model.net[5].weight.T + model.net[5].bias.unsqueeze(0)

output0 = [X[0].detach().numpy(), out1.detach().numpy()[0], out2.detach().numpy()[0], out3.detach().numpy()[0]]

In [ ]:
# print(X[0])
# print(out1)
# print(out2)
# print(out3)
# print(out4)
# print(out5)
print(output0)

In [ ]:
# plot input value changes between layers for one particular point

fig, axes = plt.subplots(2, 2, figsize=(10, 10))
axes = axes.flatten() # Flatten the 2x3 matrix to easily loop through all 6

# plot each vec
for i, ax in enumerate(axes):
    vector_len = len(output0[i])
    ax.bar(range(vector_len), output0[i], color='skyblue', edgecolor='black')
    
    ax.set_title(f'model.net[{i}][0]' if i > 0 else 'Input X[0]')
    ax.set_ylabel('Activation Value')
    ax.set_xlabel('Node Index')
    ax.grid(axis='y', alpha=0.3)

In [ ]:
# plot heatmap of weight matrix (for animation see 2nd notebook)

fig, axs = plt.subplots(1, 2, figsize=(10, 5))

im1 = axs[0].imshow(model.net[1].weight.detach().numpy().T, cmap='viridis')
axs[0].set_title('Layer 1 Weights')
fig.colorbar(im1, ax=axs[0], shrink=0.7)

im2 = axs[1].imshow(model.net[3].weight.detach().numpy().T, cmap='plasma')
axs[1].set_title('Layer 2 Weights')
fig.colorbar(im2, ax=axs[1], shrink=0.7)

# im3 = axs[2].imshow(model.net[5].weight.detach().numpy().T, cmap='coolwarm')
# axs[2].set_title('Output Layer weights')
# fig.colorbar(im3, ax=axs[2], shrink=0.7)

plt.tight_layout()
plt.show()

In [ ]:
# plot norm of input data

